# TP 2 — Eigenfaces : reconnaissance par PCA + SVM

**Objectifs**

- Calculer les eigenfaces d'un jeu de visages via PCA.
- Visualiser les premières composantes principales.
- Reconnaître l'identité d'une personne par PCA + SVM.
- Étudier l'effet du nombre de composantes sur la performance.

**Durée indicative : 60 minutes.**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

faces = fetch_olivetti_faces(shuffle=True, random_state=0)
images, y = faces.images, faces.target
h, w = images.shape[1:]
X = images.reshape(len(images), -1)
print(X.shape, "  identités :", len(np.unique(y)))

(400, 4096)   identités : 40


## Exercice 1 — PCA sur les visages

1. Faites un split train/test 75/25, stratifié par identité, `random_state=0`.
2. Fit une `PCA(n_components=150, whiten=True, random_state=0)` sur le train uniquement.
3. Transformez train et test.
4. Affichez la variance expliquée totale par les 150 composantes.

<details>
<summary>💡 Coup de pouce — PCA sur visages (Eigenfaces)</summary>

**🎯 But :** réduire 4096 pixels à ~150 composantes principales en gardant l'essentiel de l'info visage, avant un SVM.

**Charger et splitter**

```python
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
faces = fetch_olivetti_faces()
X = faces.images.reshape(400, -1)        # (400, 4096)
y = faces.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0
)
```

**PCA avec whitening**

```python
from sklearn.decomposition import PCA
pca = PCA(n_components=150, whiten=True, random_state=0).fit(X_train)
X_train_pca = pca.transform(X_train)     # (300, 150)
X_test_pca  = pca.transform(X_test)      # (100, 150)
```

**Pourquoi `whiten=True`** ?

Sans whitening, les premières composantes ont une variance énorme (capturent la luminosité globale), les dernières ont une variance minuscule. Le SVM va surpondérer les premières.

Avec `whiten=True`, chaque composante est **divisée par son écart-type** → toutes contribuent à parité. Comportement attendu pour des classifieurs basés sur la distance (SVM linéaire, k-NN).

⚠️ **Anti-leakage** : `.fit(X_train)` UNIQUEMENT, jamais sur le test. Le test utilise `.transform()` seulement, avec la PCA apprise sur le train.

**Comment 150 composantes ?**

C'est un nombre empirique pour Olivetti. La courbe variance vs n_components (Exo 4) montre que 50-150 est le bon ordre de grandeur. Moins → perd des détails, plus → ajoute du bruit.

**Variance gardée**

```python
print(f"Variance gardée : {pca.explained_variance_ratio_.sum():.1%}")
```

Typiquement **~90 % avec 150 composantes**. On a divisé la dim par 27 en gardant l'essentiel.

</details>

In [2]:
# TODO


## Exercice 2 — Visualiser les 12 premières eigenfaces

Chaque composante principale est un vecteur de longueur `h*w`. Reshape en `(h, w)` et affichez les 12 premières dans une grille 3×4.

<details>
<summary>💡 Coup de pouce — visualiser les eigenfaces</summary>

**🎯 But :** voir directement ce que la PCA a appris sur les visages. Chaque eigenface est un « visage moyen pondéré » qui capture un mode de variation.

**Récupérer les composantes**

```python
print(pca.components_.shape)   # (150, 4096)
```

`pca.components_[i]` est un **vecteur de 4096 valeurs**, identique en taille aux images aplaties. On peut le reshaper en image 64×64 et l'afficher.

**Afficher 12 eigenfaces**

```python
h, w = 64, 64
fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for i, ax in enumerate(axes.flat):
    eigenface = pca.components_[i].reshape(h, w)
    ax.imshow(eigenface, cmap='gray')
    ax.set_title(f"Eigenface #{i+1}", fontsize=9)
    ax.axis('off')
plt.suptitle("Les 12 premières directions principales (eigenfaces)")
```

**Ce que vous devriez voir**

- **Eigenface #1** : visage « moyen » + éclairage dominant (un côté plus clair).
- **Eigenface #2** : variation gauche-droite (rotation tête).
- **Eigenface #3-5** : expressions (sourire, yeux fermés, sourcils).
- **Eigenfaces #6-12** : détails plus fins (lunettes, barbe, hauteur des yeux).

Chaque visage du dataset = **combinaison linéaire de ces eigenfaces**. Le SVM apprend à reconnaître les identités à partir de ces 150 coefficients, pas des 4096 pixels.

**Pourquoi ça ressemble à des visages**

Parce que les composantes principales sont des **directions de maximum de variance** dans l'espace des pixels. Comme les images d'entrée sont toutes des visages, les directions principales encodent les façons dont les visages varient les uns des autres.

</details>

In [3]:
# TODO


## Exercice 3 — Reconnaissance d'identité

1. Entraînez `SVC(kernel="rbf", C=1e3, gamma=0.005, class_weight="balanced")` sur `X_train_pca`.
2. Évaluez sur `X_test_pca`. Affichez l'accuracy et un `classification_report` (les 40 classes seront longues, c'est ok).
3. Affichez la matrice de confusion en plus petit (sans annotations) — où sont concentrées les confusions ?

<details>
<summary>💡 Coup de pouce — classifier les identités par SVM sur eigenfaces</summary>

**🎯 But :** entraîner un SVM sur les coefficients eigenface pour reconnaître **qui est qui** parmi 40 personnes.

**Pipeline complet**

```python
from sklearn.svm import SVC
clf = SVC(kernel='linear', class_weight='balanced', C=1.0).fit(X_train_pca, y_train)
acc = clf.score(X_test_pca, y_test)
print(f"Accuracy : {acc:.3f}")
```

Sur Olivetti avec 150 composantes, attendez-vous à **0.90-0.95**.

**Pourquoi `class_weight='balanced'`** ?

Olivetti est équilibré (10 photos par personne), donc ici l'option a peu d'effet. Mais c'est un bon réflexe : sur datasets déséquilibrés, sklearn donne automatiquement plus de poids aux classes minoritaires pendant l'entraînement → évite de privilégier la classe majoritaire.

**Matrice de confusion 40×40**

```python
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
cm = confusion_matrix(y_test, clf.predict(X_test_pca))
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(cm).plot(ax=ax, include_values=False, colorbar=False)
```

⚠️ **`include_values=False`** : 40×40 = 1600 cases, mettre les chiffres dans chacune devient illisible. Pour une matrice de cette taille, on s'intéresse au **pattern visuel** : une diagonale propre = bon modèle.

**Identifier les identités ratées**

```python
import numpy as np
per_class_acc = cm.diagonal() / cm.sum(axis=1)
worst = np.argsort(per_class_acc)[:5]
print(f"5 identités les pires : {worst} (accuracy {per_class_acc[worst].round(2)})")
```

`cm.diagonal()` = bons. `cm.sum(axis=1)` = total prédit par classe. Diviser donne l'accuracy par classe.

Sur Olivetti, certaines identités à 100 %, d'autres autour de 70-80 % → souvent des visages avec **lunettes / barbe / éclairage particulier**, qui varient trop entre les 10 photos.

</details>

In [4]:
# TODO


## Exercice 4 — Courbe accuracy vs n_components

Pour `n in [10, 25, 50, 100, 150, 200]` :

- refit PCA puis SVM,
- mesurez l'accuracy test.

Tracez la courbe. Quand l'accuracy commence-t-elle à plafonner ?

<details>
<summary>💡 Coup de pouce — courbe accuracy vs n_components</summary>

**🎯 But :** trouver le nombre de composantes qui maximise la performance — ni trop peu (perd info), ni trop (ajoute bruit).

**Boucle sur n_components**

```python
results = []
for n in [10, 25, 50, 100, 150, 200, 300]:
    pca = PCA(n_components=n, whiten=True, random_state=0).fit(X_train)
    X_tr_p = pca.transform(X_train)
    X_te_p = pca.transform(X_test)
    clf = SVC(kernel='linear').fit(X_tr_p, y_train)
    acc = clf.score(X_te_p, y_test)
    results.append((n, acc))
    print(f"n={n:3d} → acc={acc:.3f}")
```

**Tracer la courbe**

```python
ns, accs = zip(*results)
plt.plot(ns, accs, marker='o')
plt.xlabel('n_components'); plt.ylabel('test accuracy')
plt.grid(True, alpha=0.3)
```

**Ce que vous devriez observer**

Une courbe qui **monte rapidement** (10 → 50) puis **sature** (50 → 150) et éventuellement **redescend légèrement** au-delà :

```
acc
1.0 │
0.9 │       . . . .      ← plateau, sweet spot
0.8 │   .              . ← peu de bruit ajouté
0.7 │ .
0.6 ┴────────────────────→
    10  50 100 150 200  300  n_components
```

**Interprétation**
- **n petit (10)** : on a jeté trop d'info. Les détails utiles pour distinguer les visages sont perdus.
- **n moyen (50-150)** : **sweet spot**. On garde l'info pertinente, on jette le bruit.
- **n grand (200+)** : les dernières composantes sont du bruit. Le SVM les utilise quand même → léger overfit, perf qui dégrade.

**À retenir**

PCA fait **deux choses** : réduit la dim **et régularise** (en jetant les composantes de faible variance, qui sont souvent du bruit). C'est pourquoi un peu de PCA améliore souvent les modèles linéaires, même si on garde toujours beaucoup d'info.

</details>

In [5]:
# TODO
